<a href="https://colab.research.google.com/github/SarahkhIT/DataEngProject/blob/main/notebooks/04_orchestration_airflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4. Orchestration — Airflow DAG

Wires every stage together with correct dependencies. If the Quality Gate task fails, Airflow halts everything downstream — nothing after it runs.

In [ ]:
!airflow db migrate

2026-07-22T23:17:54.332346Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////root/airflow/airflow.db
2026-07-22T23:17:54.553316Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-07-22T23:17:54.553771Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-07-22T23:17:54.553968Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-07-22T23:17:54.554125Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-07-22T23:17:54.554273Z [info     ] setup plugin alembic.autogenerate.defaults [alembic.runtime.plugins] loc=plugins.py:37
2026-07-22T23:17:54.554419Z [info     ] setup plugin alembic.autogenerate.comments [alembic.runtime.plugins] loc=plugins.py:37
2026-07-22T23:17:54.594460Z [info     ] Context impl SQLiteImp

In [ ]:
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime

with DAG(
    dag_id="arxiv_research_pipeline",
    start_date=datetime(2026, 7, 22),
    schedule=None,
    catchup=False,
) as dag:

    t1 = PythonOperator(task_id="pydantic_validation", python_callable=run_pydantic_validation)
    t2 = PythonOperator(task_id="kafka_ingestion", python_callable=run_kafka_producer)
    t3 = PythonOperator(task_id="kafka_consumer_dlq", python_callable=run_kafka_consumer)
    t4 = PythonOperator(task_id="bronze_layer", python_callable=save_bronze)
    t5 = PythonOperator(task_id="quality_gate", python_callable=run_ge_gate)
    t6 = PythonOperator(task_id="silver_layer", python_callable=build_silver)
    t7 = PythonOperator(task_id="gold_layer", python_callable=build_gold)

    t1 >> t2 >> t3 >> t4 >> t5 >> t6 >> t7

print("✅ DAG defined:", dag.dag_id)

✅ DAG defined: arxiv_research_pipeline


In [ ]:
import os

AIRFLOW_HOME = os.environ.get("AIRFLOW_HOME", "/root/airflow")
dags_folder = os.path.join(AIRFLOW_HOME, "dags")
os.makedirs(dags_folder, exist_ok=True)

dag_code = '''
import pandas as pd
import json
import os
from datetime import datetime
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from deltalake import DeltaTable
from deltalake.writer import write_deltalake

bronze_layer_path = "delta_lakehouse/bronze/arxiv_data"
silver_path = "delta_lakehouse/silver/arxiv_data"

def task_bronze_layer():
    df_bronze = pd.read_csv("arxiv_valid_bronze.csv", dtype=str)
    write_deltalake(bronze_layer_path, df_bronze, mode="overwrite")
    print(f"Wrote {len(df_bronze)} records to Bronze.")

def task_quality_gate():
    import great_expectations as gx
    import great_expectations.expectations as gxe
    df = DeltaTable(bronze_layer_path).to_pandas()
    context = gx.get_context()
    asset = context.data_sources.add_pandas("papers").add_dataframe_asset(name="asset")
    batch = asset.build_batch_request(options={"dataframe": df})
    suite = gx.ExpectationSuite(name="suite")
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="title"))
    suite.add_expectation(gxe.ExpectColumnValuesToMatchRegex(column="category", regex=r"^cs\\\\."))
    result = context.get_validator(batch_request=batch, expectation_suite=suite).validate()
    if not result["success"]:
        raise RuntimeError("Quality gate failed")
    print("Quality gate passed.")

def task_silver_layer():
    df = DeltaTable(bronze_layer_path).to_pandas().drop_duplicates(subset=["paper_id"])
    write_deltalake(silver_path, df, mode="overwrite")
    print(f"Silver has {len(df)} records.")

def task_gold_layer():
    df = DeltaTable(silver_path).to_pandas()
    gold = df.groupby("category").size().reset_index(name="count")
    write_deltalake("delta_lakehouse/gold/papers_by_category", gold, mode="overwrite")
    print("Gold written.")

with DAG(
    dag_id="arxiv_research_pipeline",
    start_date=datetime(2026, 7, 22),
    schedule=None,
    catchup=False,
) as dag:
    t4 = PythonOperator(task_id="bronze_layer", python_callable=task_bronze_layer)
    t5 = PythonOperator(task_id="quality_gate", python_callable=task_quality_gate)
    t6 = PythonOperator(task_id="silver_layer", python_callable=task_silver_layer)
    t7 = PythonOperator(task_id="gold_layer", python_callable=task_gold_layer)
    t4 >> t5 >> t6 >> t7
'''

with open(os.path.join(dags_folder, "capstone_pipeline_dag.py"), "w") as f:
    f.write(dag_code)

print(f"✅ DAG file written to {dags_folder}/capstone_pipeline_dag.py")

✅ DAG file written to /root/airflow/dags/capstone_pipeline_dag.py


In [ ]:
import subprocess
result = subprocess.run(
    ["airflow", "dags", "test", "arxiv_research_pipeline", "2026-07-22"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-2000:])

r.py", line 2125, in _execute_task
    result = ctx.run(execute, context=context)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/airflow/sdk/bases/operator.py", line 445, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/airflow/providers/standard/operators/python.py", line 231, in execute
    return_value = self.execute_callable()
                   ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/airflow/providers/standard/operators/python.py", line 254, in execute_callable
    return runner.run(*self.op_args, **self.op_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/airflow/sdk/execution_time/callback_runner.py", line 97, in run
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/root/airflow/dags/capstone_pipeline_dag.py", line 31, in task_qu

In [ ]:
def failing_quality_gate():
    raise RuntimeError("Intentional quality gate failure for rubric proof")

with DAG(
    dag_id="arxiv_quality_gate_failure_proof",
    start_date=datetime(2026, 7, 22),
    schedule=None,
    catchup=False,
) as failure_dag:

    t1 = PythonOperator(task_id="bronze_layer", python_callable=save_bronze)
    t2 = PythonOperator(task_id="quality_gate_should_fail", python_callable=failing_quality_gate)
    t3 = PythonOperator(task_id="silver_layer_should_not_run", python_callable=build_silver)
    t4 = PythonOperator(task_id="gold_layer_should_not_run", python_callable=build_gold)

    t1 >> t2 >> t3 >> t4

print("Failure-proof DAG defined:", failure_dag.dag_id)

Failure-proof DAG defined: arxiv_quality_gate_failure_proof


In [ ]:
result = subprocess.run(
    ["airflow", "dags", "test", "arxiv_quality_gate_failure_proof", "2026-07-22"],
    capture_output=True,
    text=True,
)

print(result.stdout[-5000:])
print(result.stderr[-3000:])

ta_interval_end=None next_dagrun_data_interval_start=None next_dagrun_partition_date=None next_dagrun_partition_key=None
2026-07-22T23:24:15.588594Z [info     ] get next_dagrun_info_v2        [airflow.serialization.definitions.dag] last_automated_run_info=None loc=dag.py:445 next_info=None
2026-07-22T23:24:15.588809Z [info     ] setting next dagrun info       [airflow.models.dag] loc=dag.py:848 next_dagrun=None next_dagrun_create_after=None next_dagrun_data_interval_end=None next_dagrun_data_interval_start=None next_dagrun_partition_date=None next_dagrun_partition_key=None
2026-07-22T23:24:15.590371Z [info     ] get next_dagrun_info_v2        [airflow.serialization.definitions.dag] last_automated_run_info=None loc=dag.py:445 next_info=None
2026-07-22T23:24:15.590551Z [info     ] setting next dagrun info       [airflow.models.dag] loc=dag.py:848 next_dagrun=None next_dagrun_create_after=None next_dagrun_data_interval_end=None next_dagrun_data_interval_start=None next_dagrun_partition_da

**Orchestration complete** — DAG runs every stage in dependency order; a failed quality gate halts downstream tasks.